# Serie I — fuente, forma del archivo y unidad de `T_SPACRE`

## Decisión documentada

El PDF de la prueba llama a `T_SPACRE` toneladas de azúcar por acre. La ficha oficial USDA-ARS define esta variable como libras de azúcar por acre. Para el análisis usaremos la unidad de USDA y anotamos aquí la diferencia para que sea verificable.

El CSV de USDA se descarga con `python scripts/descargar_serie1.py`. Los datos quedan en `data/raw/`, fuera de Git. La fuente oficial es [Data.gov](https://catalog.data.gov/dataset/louisiana-sugarcane-yield-data-2010-2021-collected-for-the-louisiana-variety-development-p) y el archivo asociado se sirve desde [Ag Data Commons / Figshare](https://ndownloader.figshare.com/files/49874100).

In [1]:
from pathlib import Path
import hashlib
import pandas as pd

file_name = "louisiana_sugarcane_2010_2021.csv"
candidates = [Path.cwd() / "data" / "raw" / file_name, Path.cwd().parent / "data" / "raw" / file_name]
csv_path = next((path for path in candidates if path.exists()), None)
if csv_path is None:
    raise FileNotFoundError("No se encontró el CSV. Desde la carpeta del proyecto ejecuta: python scripts/descargar_serie1.py")

file_hash = hashlib.sha256(csv_path.read_bytes()).hexdigest()
raw = pd.read_csv(csv_path, dtype=str, keep_default_na=False)
print(f"Archivo: data/raw/{csv_path.name}")
print(f"SHA-256: {file_hash}")
print(f"Filas de datos: {len(raw):,}; columnas físicas: {raw.shape[1]}")

Archivo: data/raw/louisiana_sugarcane_2010_2021.csv
SHA-256: 5e7b13de6bda8aed51479c5fdfc3de4a28e71ccbb45e4761b6688a34b7cff4b0
Filas de datos: 29,529; columnas físicas: 15


## Estructura física del CSV

La primera docena de campos corresponde a las variables del ensayo. También inspecciono las columnas finales, que en el archivo no tienen nombre y contienen anotaciones sueltas. Esto evita que una nota de diccionario termine tratada como medición.

In [2]:
core_columns = [
    "CLONE", "YEAR", "STAGE", "LOC", "CROP", "REP",
    "T_SPACRE", "TRS_TON", "TCA", "POPN", "MSTWT", "FIBER",
]
unnamed_columns = list(raw.columns[12:])
annotations = raw.iloc[:, 12:].replace("", pd.NA)
print("Variables analíticas:", core_columns)
print("Columnas finales sin nombre:", unnamed_columns)
print("Cantidad de valores presentes en las columnas finales:")
display(annotations.count().to_frame(name="valores no vacíos"))
print("Anotaciones que aparecen en esas columnas:")
display(annotations.stack().dropna().to_frame(name="anotación"))

Variables analíticas: ['CLONE', 'YEAR', 'STAGE', 'LOC', 'CROP', 'REP', 'T_SPACRE', 'TRS_TON', 'TCA', 'POPN', 'MSTWT', 'FIBER']
Columnas finales sin nombre: ['Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14']
Cantidad de valores presentes en las columnas finales:


,valores no vacíos
Unnamed: 12,0
Unnamed: 13,0
Unnamed: 14,6


Anotaciones que aparecen en esas columnas:


,,anotación
0,Unnamed: 14,"T_SPACRE, ons of sugar per acre"
1,Unnamed: 14,"Rep, replication"
2,Unnamed: 14,"TRS, theoretical recoverable sugar"
3,Unnamed: 14,"TCA, ton of cane per acre"
4,Unnamed: 14,"POPN, population"
5,Unnamed: 14,"MSTWT, mean stalk weight"


## Comprobaci?n dimensional entre rendimiento y calidad

Si `TCA` est? en toneladas de ca?a por acre y `TRS_TON` en libras de az?car por tonelada de ca?a, su producto tiene unidades de libras de az?car por acre. El CSV contiene celdas simples y celdas con dos componentes num?ricos separados por espacios. Conservo el texto original, separo el prefijo sin interpretarlo y dejo en `TRS_TON` el componente num?rico final. Comparo `TCA ? TRS_TON` con `T_SPACRE` en filas positivas y completas, con tolerancia relativa de 1 %. Esta comprobaci?n respalda la coherencia interna; la unidad oficial procede de la ficha USDA. El significado del prefijo est? pendiente de validaci?n documental.


In [3]:
trs_raw = raw["TRS_TON"].str.strip().replace({"": pd.NA, ".": pd.NA})
trs_parts = trs_raw.str.extract(r"^(?:(?P<prefijo>\d+)\s+)?(?P<componente_final>\d+(?:\.\d+)?)$")
measurements = raw[core_columns].copy()
measurements["TRS_TON_raw"] = raw["TRS_TON"]
measurements["TRS_TON_prefijo_sin_interpretar"] = trs_parts["prefijo"].astype("string")
measurements["TRS_TON"] = pd.to_numeric(trs_parts["componente_final"], errors="coerce")
for column in ["T_SPACRE", "TCA"]:
    measurements[column] = pd.to_numeric(
        measurements[column].replace("", pd.NA).replace(".", pd.NA), errors="coerce"
    )

measure_columns = ["T_SPACRE", "TRS_TON", "TCA"]
complete_positive = measurements.dropna(subset=measure_columns).copy()
complete_positive = complete_positive.loc[(complete_positive[measure_columns] > 0).all(axis=1)].copy()
complete_positive["azúcar estimada (lb/acre)"] = complete_positive["TCA"] * complete_positive["TRS_TON"]
complete_positive["razón observado / calculado"] = complete_positive["T_SPACRE"] / complete_positive["azúcar estimada (lb/acre)"]
agrees = complete_positive["razón observado / calculado"].between(0.99, 1.01)

summary = pd.DataFrame({
    "resultado": [
        len(raw), int(measurements["TRS_TON_prefijo_sin_interpretar"].notna().sum()),
        len(complete_positive), int(agrees.sum()), float(agrees.mean() * 100),
        float(complete_positive["razón observado / calculado"].median()),
    ]
}, index=[
    "filas de datos", "celdas TRS con dos componentes", "filas positivas completas",
    "filas que coinciden dentro de 1 %", "coincidencia (%)", "mediana de razón observado/calculado",
])
display(summary)

exceptions = complete_positive.loc[
    ~agrees,
    core_columns + ["TRS_TON_raw", "TRS_TON_prefijo_sin_interpretar", "azúcar estimada (lb/acre)", "razón observado / calculado"],
]
display(exceptions.sort_values("razón observado / calculado").reset_index(names="fila_csv"))

,resultado
filas de datos,29529.000000
celdas TRS con dos componentes,10485.000000
filas positivas completas,29340.000000
filas que coinciden dentro de 1 %,29317.000000
coincidencia (%),99.921609
mediana de razón observado/calculado,0.999975


,fila_csv,CLONE,YEAR,STAGE,LOC,CROP,REP,T_SPACRE,TRS_TON,TCA,POPN,MSTWT,FIBER,TRS_TON_raw,TRS_TON_prefijo_sin_interpretar,azúcar estimada (lb/acre),razón observado / calculado
0,9154,L99-226,2013,2,SGH,1,1,14689.0,205.500,74.0,60121.8,2.405,11.65,19 205.500,19,15207.0000,0.965937
1,9270,L01-299,2011,2,SGH,0,1,6875.0,221.395,32.1,29040,2.158,11.6,84 221.395,84,7106.7795,0.967386
2,15085,20069610,2010,3,STG,2,2,343.0,220.790,1.6,4083.8,0.76,,220.79,<NA>,353.2640,0.970945
3,23629,HOCP00-950,2012,2,SGH,2,1,8713.0,270.610,33.1,38115,1.642,,91 270.610,91,8957.1910,0.972738
4,29521,HOCP00-950,2016,5,ALLAINS,1,3,706.0,313.714,2.3,10467.6,0.43,,02 313.714,02,721.5422,0.978460
5,9824,HOCP00-950,2010,2,SGH,0,1,11176.0,268.015,42.3,30855,2.7,11.36,48 268.015,48,11337.0345,0.985796
6,12320,L03-371,2013,2,SGH,1,1,7906.0,202.000,39.7,40156.8,1.992,9.46,30 202.000,30,8019.4000,0.985859
7,6355,L99-226,2016,2,SGH,1,1,4720.0,247.998,19.3,30628.1,1.245,12.969089,18 247.998,18,4786.3614,0.986135
8,10988,HOCP96-540,2013,2,SGH,0,1,5506.0,169.000,33.0,29266.8,2.25,10.75,90 169.000,90,5577.0000,0.987269
9,7408,L99-233,2011,2,SGH,1,1,6864.0,204.405,34.0,42879.3,1.575,12.4308,11 204.405,11,6949.7700,0.987659


## Conclusión y alcance

El catálogo oficial de USDA-ARS declara `T_SPACRE` en **libras de azúcar por acre**. El archivo descargado de la dirección oficial y el archivo enlazado por Kaggle en el PDF tienen SHA-256 idéntico (`5e7b13de6bda8aed51479c5fdfc3de4a28e71ccbb45e4761b6688a34b7cff4b`), por lo que son el mismo CSV. En 10,485 celdas, el valor fuente de `TRS_TON` contiene dos componentes numéricos. El componente final da coherencia dimensional dentro de 1 % en 29,317 de 29,340 filas positivas completas (99.92 %); 23 filas quedan fuera de tolerancia y el prefijo sigue sin explicación. El ETL conserva el valor original en `TRS_TON_raw`, separa el prefijo y deja el componente numérico final en `TRS_TON`. La unidad absoluta se adopta de USDA y la discrepancia del PDF está explicada al comienzo del README.

Esta auditoría resuelve la unidad documentada de la variable, no valida aún todo el dataset ni convierte las mediciones de Louisiana en pronósticos confiables para Santa Ana. Las columnas auxiliares, valores faltantes, diseño de ensayos y calidad estadística se revisan en el perfil EDA antes de elegir variedades o modelos.